# tHDA primer design

Give it a **template sequence** and it returns one **forward / reverse primer pair** that meets the NEB tHDA recommendations:

| | min | opt | max |
|---|---|---|---|
| Product size (bp) | 80 | 100 | 120 |
| Product Tm (°C) | 68 | 71 | 75 |
| Primer size (nt) | 24 | 27 | 33 |
| Primer Tm (°C) | 60 | 68 | 74 |
| Primer GC (%) | 35 | 44 | 60 |

The optima shift with the template's GC content as NEB specifies (primer 26 nt / Tm 71 °C above 45 % GC, Tm 65 °C below 37.5 % GC). Tm values use the original Primer3 model the NEB ranges were calibrated with.

**Optionally** give a whole-genome FASTA: the primers are then BLASTed against *that genome only*, to show every other place they could bind and any extra product they could make.

In [1]:
from hda_primer_design import (HDAParams, design_hda_primers, design_for_locus,
                               gc_content, neb_check, format_neb_check,
                               read_fasta, write_table)
from hda_blast import BlastConfig, blast_db_from_fasta, scan_genome, summarize_genome_scan

## 1. Input
`LOCUS` is the region to amplify, as 1-based inclusive genome coordinates; the template is that slice. `GENOME` is also used in step 4 to find any other place the primers could bind — set it to `None` to skip that.

In [2]:
GENOME = 'e-coli-mg1655-genome.fasta'     # whole genome; set to None to skip the genome BLAST
LOCUS = (2878409, 2878508)                # region to amplify, 1-based inclusive

genome_seq = read_fasta(GENOME)[0][1]
TEMPLATE = genome_seq[LOCUS[0] - 1:LOCUS[1]]

# Or give a template directly and leave GENOME = None:
# TEMPLATE, LOCUS, genome_seq = 'ACGT...', None, None

print(f'Genome:   {len(genome_seq):,} bp, GC {gc_content(genome_seq):.1f}%')
print(f'Template: {LOCUS[0]}-{LOCUS[1]} = {len(TEMPLATE)} bp, GC {gc_content(TEMPLATE):.1f}%')
print(TEMPLATE)

Genome:   4,641,652 bp, GC 50.8%
Template: 2878409-2878508 = 100 bp, GC 47.0%
AGCGTATCGCCGCGCGTCTGCGAAAGCGGTTTATCCCCGCTGGCGCGGGGAACTCTCTAAAAGTATACATTTGTTCTTAAAGCATTTTTTCCCATAAAAA


## 2. Primers
`design_hda_primers` tiles primer3 along the template and returns the best-scoring pair (primer3 penalty, plus a preference for ~40 % amplicon GC and for primer GC close to the amplicon's). Every value is then checked against the NEB ranges.

In [3]:
params = HDAParams.for_target(gc_content(TEMPLATE))
pairs, info = design_hda_primers(TEMPLATE, params=params, n_regions=1, pairs_per_region=1)

if not pairs:
    print(f'No NEB-compliant pair fits inside these {len(TEMPLATE)} bp. primer3 reports:')
    for e in info['primer3_explain_when_empty'][:1]:
        for part in e.split('; '):
            print('   ', part)
    print('\nGC along the template (20 bp windows):')
    print('   ', '  '.join(f'{gc_content(TEMPLATE[j:j+20]):.0f}%'
                           for j in range(0, len(TEMPLATE), 20)))
    print('\n-> use step 3 to search the flanking sequence as well.')
else:
    pair = pairs[0]
    print(f'Forward  {pair.left_seq}   {pair.left_len} nt, Tm {pair.left_tm:.1f} C, GC {pair.left_gc:.0f}%')
    print(f'Reverse  {pair.right_seq}   {pair.right_len} nt, Tm {pair.right_tm:.1f} C, GC {pair.right_gc:.0f}%')
    print(f'Product  {pair.product_size} bp at template {pair.left_start + 1}-{pair.right_end + 1}, '
          f'Tm {pair.product_tm:.1f} C, GC {pair.amplicon_gc:.1f}%')
    print('\nNEB tHDA criteria:')
    print(format_neb_check(neb_check(pair, params)))

No NEB-compliant pair fits inside these 100 bp. primer3 reports:
    left: considered 210, GC content failed 176, high tm 28, ok 6
    right: considered 210, GC content failed 210, ok 0
    pair: considered 0, ok 0

GC along the template (20 bp windows):
    70%  60%  60%  20%  25%

-> use step 3 to search the flanking sequence as well.


## 3. If the template alone is too short or too uneven in GC
A 100 bp template leaves no room: the product must be 80-120 bp and both primers sit inside it, so primers are forced onto the template's ends. If those ends are GC-extreme, no compliant pair exists.

`design_for_locus` searches the locus **plus `FLANK` bp either side** and keeps only pairs whose amplicon still overlaps the locus, ranked by how much of it they cover. Coordinates are absolute genome positions.

In [4]:
FLANK = 300        # bp searched either side of the locus
MIN_OVERLAP = 1    # minimum bp of the locus the amplicon must cover

locus_pairs, locus_info = design_for_locus(genome_seq, *LOCUS, flank=FLANK,
                                           min_overlap=MIN_OVERLAP, n_regions=8)
print(f'Searched {locus_info["searched"][0]:,}-{locus_info["searched"][1]:,}; '
      f'{locus_info["n_overlapping"]} compliant pair(s) overlap the locus\n')
for q in locus_pairs:
    print(f'#{q.region}  {q.left_start + 1:,}-{q.right_end + 1:,}  {q.product_size} bp, '
          f'product Tm {q.product_tm:.1f} C  |  covers {q.locus_overlap}/{LOCUS[1] - LOCUS[0] + 1} bp '
          f'of the locus ({q.locus_covered:.0f}%)')
    print(f'    F {q.left_seq}  ({q.left_len} nt, Tm {q.left_tm:.1f}, GC {q.left_gc:.0f}%)')
    print(f'    R {q.right_seq}  ({q.right_len} nt, Tm {q.right_tm:.1f}, GC {q.right_gc:.0f}%)')

# Use the best locus-overlapping pair from here on, if the template itself gave none.
if not pairs and locus_pairs:
    pair = locus_pairs[0]
    print('\nNEB tHDA criteria for #1:')
    print(format_neb_check(neb_check(pair, locus_info['params'])))

Searched 2,878,109-2,878,808; 1 compliant pair(s) overlap the locus

#1  2,878,448-2,878,537  90 bp, product Tm 68.7 C  |  covers 61/100 bp of the locus (61%)
    F CTGGCGCGGGGAACTCTCTAAAAGTA  (26 nt, Tm 69.9, GC 54%)
    R GGAAATGTTACATTAAGGTTGGTGGGTTG  (29 nt, Tm 67.6, GC 41%)

NEB tHDA criteria for #1:
  ok   Product size (bp)                        90   (80-120)
  ok   Product Tm (C)                         68.7   (68-75)
  ok   Forward length (nt)                      26   (24-33)
  ok   Reverse length (nt)                      29   (24-33)
  ok   Forward Tm (C)                         69.9   (60-74)
  ok   Reverse Tm (C)                         67.6   (60-74)
  ok   Forward GC (%)                         53.8   (35-60)
  ok   Reverse GC (%)                         41.4   (35-60)
  ok   Primer Tm difference (C)                2.3   (<= 5)
  --   Amplicon GC (%)                        37.8   (~40 preferred)
  --   Mean primer GC vs amplicon GC (%)       9.8   (small preferred)


## 4. Where else could the primers bind? (optional)
Runs only if `GENOME` is set. A BLAST database is built from that one genome (reused on later runs) and both primers are searched in it.

A **binding site** is a place where the primer's 3′ end matches exactly and the full-length primer has at most `max_mismatches` mismatches — the 5′ end may mismatch, since that still primes, but a 3′-terminal mismatch blocks extension. Mismatches are counted base by base against the genome, not taken from the BLAST alignment, so partially aligned sites are not missed. Two facing sites within `max_product` bp are reported as a product.

In [5]:
if GENOME is None:
    print('GENOME is None - skipping the genome BLAST.')
else:
    db = blast_db_from_fasta(GENOME)           # or pass out_prefix='/path/to/db'
    sites, products = scan_genome(pair, BlastConfig(db=db, threads=8),
                                  max_mismatches=3,   # per primer, over its full length
                                  max_product=3000)   # largest product to report
    print(summarize_genome_scan(sites, products, max_mismatches=3))

Built BLAST database e-coli-mg1655-genome_blastdb


Forward  CTGGCGCGGGGAACTCTCTAAAAGTA     2 binding site(s) with <= 3 mismatches, 1 perfect
    E:2878448 (+) 0 mm, 26/26 nt aligned
    E:2904392 (+) 3 mm, 23/26 nt aligned
Reverse  GGAAATGTTACATTAAGGTTGGTGGGTTG  1 binding site(s) with <= 3 mismatches, 1 perfect
    E:2878537 (-) 0 mm, 29/29 nt aligned
Products: 1 intended, 0 additional
    INTENDED E:2878448-2878537 90 bp, F/R, 0 mm


## 5. Save
Order **desalted / salt-free** primers: tHDA is sensitive to salt.

In [6]:
chosen = pairs or locus_pairs
write_table(chosen, 'hda_primers.tsv')
print(f'wrote hda_primers.tsv ({len(chosen)} pair(s))')

wrote hda_primers.tsv (1 pair(s))


---
Beyond this notebook, the modules also cover: specificity across all bacteria (`hda_blast.check_specificity` against `ref_prok_rep_genomes`), strain-specific amplicon discovery (`hda_unique.design_strain_specific`), and longer/shorter primer variants around a working set (`hda_primer_design.serial_primer_variants`).